In [ ]:
# imports
import os
import requests
from typing import Optional
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, DisplayHandle, display
from openai import OpenAI, OpenAIError
from openai.types.chat.chat_completion import ChatCompletion

In [ ]:
# load api key
load_dotenv()
api_key: Optional[str] = os.getenv("OPEN_AI_KEY")

if not api_key:
    raise ValueError("API_KEY does not exist.")
elif api_key[:8] != 'sk-proj-':
    raise ValueError("Invalid Key.")
elif api_key.strip() != api_key:
    raise ValueError("Invalid Key.")
else:
    print("OpenAI API key loaded successfully.")

In [ ]:
# loading the OpenAI client
try:
    client = OpenAI(api_key=api_key)
except OpenAIError as e:
    raise RuntimeError(f"Failed to initialize OpenAI client: {e}")

**Web Scrapper Class**

In [ ]:
class WebScraper:
    url: str
    title: Optional[str] = None
    content: str

    def __init__(self, url: str):
        self.url = url
        self.__fetch_content()

    def __fetch_content(self) -> None:
        """
        Fetches the content of the web page at the specified URL.
        Parses the HTML to extract the title and main content, removing scripts, styles, images, and inputs.
        :return: None
        """
        response = requests.get(self.url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title and soup.title.string else "No title found"
        for tag in soup(["script", "style", "img", "input"]):
            tag.decompose()
        self.content = soup.get_text(separator="\n", strip=True)

    def get_title(self) -> Optional[str]:
        """
        Returns the title of the web page.
        If the title is not found, returns a default message.
        :return: Title of the web page.
        """
        return self.title
    
    def get_content(self) -> str:
        """
        Returns the main content of the web page.
        The content is cleaned of scripts, styles, images, and inputs.
        :return: Cleaned content of the web page.
        """
        return self.content
    
    def get_markdown_content(self, content: Optional[str]) -> Optional[DisplayHandle]:
        """
        Returns the content formatted as Markdown.
        :return: DisplayHandle for the Markdown content.
        """
        if not content:
            content = self.get_content()
        markdown_content = f"# {self.get_title()}\n\n{content}"
        return display(Markdown(markdown_content))

# Types of Prompts
1. **System Prompt:** tells the LLM what task it is performing and what tone it should use

2. **User Prompt** the conversation from the user that they should reply to

In [ ]:
system_prompt = "You are a helpful assistant. \
You task is to analyze the content of the web page and provide a short summary in markdow. \
You should use a professional tone and be concise. \
You should not include any personal opinions or subjective statements. \
Ignore the content related to the website navigation"

In [ ]:
# User Prompt

def user_prompt(website: WebScraper) -> str:
    """
    Generates a user prompt based on the content of the web page.
    :param website: WebScraper instance containing the web page content.
    :return: User prompt string.
    """
    
    prompt = f"You are looking at the content of the website titled as '{website.get_title()}'. \
    Your task is to analyze the content and provide a short summary in markdown. \
    The content is as follows:\n\n{website.get_content()}\n\n"
    return prompt

# Messages
```python
[
    {"role": "system", "content": "system message"},
    {"role": "user", "content": "user message"}
]
```

In [ ]:
def messages_for_llm(website: WebScraper) -> list[dict[str, str]]:
    """
    Creates a list of messages for the LLM.
    :param website: WebScraper instance containing the web page content.
    :return: List of messages formatted for the LLM.
    """
    
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt(website)}
    ]

# Summarize via LLM

In [ ]:
def summarize_website(website: WebScraper) -> Optional[str]:
    """
    Summarizes the content of the web page using the LLM.
    :param website: WebScraper instance containing the web page content.
    :return: DisplayHandle for the Markdown summary.
    """
    try:
        response: ChatCompletion = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages_for_llm(website),
            max_tokens=500,
            temperature=0.7
        )
        return response.choices[0].message.content
    except OpenAIError as e:
        raise RuntimeError(f"Failed to summarize website: {e}")

web_scapper = WebScraper("https://faizanpervaiz.com/")
web_scapper.get_markdown_content(
    summarize_website(web_scapper)
)